# Gemma 4 harness backend on Colab (T4-tuned)

Serves the **same** llama.cpp build (`b11696`) and the **same** Q4_0 GGUF as the laptop, so a
trace collected here is the same experiment. The laptop runs the agent loop; this notebook only
provides the OpenAI-compatible endpoint, published with a `cloudflared` quick tunnel.

**Runtime → Change runtime type → T4 GPU.** (L4/A100 also work and allow more slots.)

This notebook is **safe to re-run**: it kills any stale server/tunnel, picks a free port (the
earlier failure was port 8080 already bound), restores the build and weights from a Google Drive
cache so a runtime restart is fast, and auto-sizes the number of server slots to your VRAM.

Run top to bottom (`Runtime → Run all`). The last cells print `BASE_URL` and `API_KEY`.

> Parity rule: **never lower `CTX_PER_SLOT`** (32768). It must equal the laptop server's `-c`,
> otherwise long prompts get truncated and the agent behaves differently. We scale *slots* to fit
> VRAM, not the per-request window.

## 0. GPU check

In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1 || echo 'no nvcc'

In [ ]:
import os, subprocess, time, secrets, socket
import psutil
import torch

MODEL = "E2B"            # "E2B" or "E4B"
CTX_PER_SLOT = 32768     # MUST equal the laptop server's -c (do not lower)
LLAMA_TAG = "b11696"     # the release the laptop's binaries came from
API_KEY = secrets.token_urlsafe(24)
HF_TOKEN = ""            # set only if the GGUF repo is gated for your account

major, minor = torch.cuda.get_device_capability()
ARCH = f"{major}{minor}"
gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {torch.cuda.get_device_name(0)}  sm_{ARCH}  {gib:.1f} GiB")

# Safe to re-run: drop any stale server/tunnel, then take a free port.
subprocess.run(["bash", "-lc", 'pkill -9 -f "build/bin/llama-server" || true'], capture_output=True)
subprocess.run(["bash", "-lc", 'pkill -9 -f "cloudflared" || true'], capture_output=True)
time.sleep(2)

def free_port(start=8080):
    for port in range(start, start + 50):
        with socket.socket() as s:
            try:
                s.bind(("127.0.0.1", port))
                return port
            except OSError:
                continue
    raise RuntimeError("no free port in range")

PORT = free_port()

# Slots are limited by HOST RAM, not VRAM. Measured on this build: the 8-slot server that got
# killed held ~1.27 GiB of host RSS per slot (dmesg anon-rss 10186332kB / 8) on a 12.97 GiB VM
# that also runs Colab's own node + language-server. Budget by both RAM (2 GiB/slot, headroom)
# and VRAM (~1 GiB of KV per 32768-token slot, since 8 slots fit a 14.6 GiB T4).
ram_gib  = psutil.virtual_memory().total / 2**30
MAX_SLOTS = max(1, min(8, int((gib - 3.0) // 1.0), int((ram_gib - 3.0) // 2.0)))
print(f"PORT = {PORT}   VRAM = {gib:.1f} GiB   RAM = {ram_gib:.1f} GiB   MAX_SLOTS = {MAX_SLOTS}")

## 0b. Drive cache (optional but recommended)

Caches the compiled `llama-server` and the GGUF in your Drive, so a runtime restart skips the
~15 min build and the 3.4 GB download. Set `USE_DRIVE_CACHE = False` to skip.

In [ ]:
USE_DRIVE_CACHE = True
DRIVE_DIR = "/content/drive/MyDrive/gemma4_colab_cache"

if USE_DRIVE_CACHE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print("cache dir:", DRIVE_DIR)
        print("contents:", os.listdir(DRIVE_DIR))
    except Exception as e:
        USE_DRIVE_CACHE = False
        print("Drive cache disabled:", type(e).__name__, e)

## 1. Build llama.cpp (CUDA, cached)

Pinned to the laptop's tag. ~10–25 min on a first build (2 vCPU); instant if restored from Drive.

In [ ]:
BIN = "llama.cpp/build/bin/llama-server"
CACHE = f"{DRIVE_DIR}/llama-{LLAMA_TAG}-cuda{ARCH}-bin.tar.gz" if USE_DRIVE_CACHE else None

if not os.path.exists(BIN) and CACHE and os.path.exists(CACHE):
    print("restoring build from Drive cache ...")
    os.makedirs("llama.cpp", exist_ok=True)
    !tar xzf "$CACHE" -C llama.cpp

if not os.path.exists(BIN):
    print(f"building llama.cpp {LLAMA_TAG} for sm_{ARCH} ...")
    if not os.path.exists("llama.cpp/CMakeLists.txt"):
        !git clone --depth 1 --branch $LLAMA_TAG https://github.com/ggml-org/llama.cpp
    !cd llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=$ARCH -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release
    !cd llama.cpp && cmake --build build --target llama-server -j 2
    if CACHE:
        !cd llama.cpp && tar czf "$CACHE" build/bin
        print("cached build ->", CACHE)
else:
    print("llama-server ready:", BIN)

!ls -la llama.cpp/build/bin/llama-server

## 2. Weights (cached)

Same repo the laptop's `scripts/setup_inference.ps1` uses: `google/gemma-4-<M>-it-qat-q4_0-gguf`.

In [ ]:
import shutil
from huggingface_hub import hf_hub_download, login

fname = f"gemma-4-{MODEL}_q4_0-it.gguf"
os.makedirs("models", exist_ok=True)
GGUF = os.path.join("models", fname)
CACHE_GGUF = f"{DRIVE_DIR}/{fname}" if USE_DRIVE_CACHE else None

if not os.path.exists(GGUF) and CACHE_GGUF and os.path.exists(CACHE_GGUF):
    print("copying weights from Drive ...")
    shutil.copy(CACHE_GGUF, GGUF)

if not os.path.exists(GGUF):
    if HF_TOKEN:
        login(HF_TOKEN, add_to_git_credential=False)
    print("downloading weights ...")
    hf_hub_download(f"google/gemma-4-{MODEL}-it-qat-q4_0-gguf", fname, local_dir="models")
    if CACHE_GGUF:
        shutil.copy(GGUF, CACHE_GGUF)

print("gguf:", GGUF, f"{os.path.getsize(GGUF) / 2**30:.2f} GiB")

## 3. Serve — auto-sized to VRAM

Same flags as `scripts/serve.ps1` (`-ngl 99 --jinja --flash-attn on -c … --metrics`) plus `-np`.
Tries decreasing slot counts and falls back from `--flash-attn on` to `auto` (Turing is picky).

A T4 (15 GB) fits about **3 slots** at 32768/slot; L4 ≈ 4–5; A100 = 8. **Note the winning `SLOTS`**
— the laptop uses `--workers <that number>`.

In [ ]:
import subprocess, time, requests

def try_serve(slots, fa):
    logpath = f"server_{slots}_{fa}.log"
    p = subprocess.Popen(["./llama.cpp/build/bin/llama-server", "-m", GGUF,
                          "-c", str(CTX_PER_SLOT), "-ngl", "99", "--jinja",
                          "--flash-attn", fa,
                          "--host", "127.0.0.1", "--port", str(PORT), "--metrics",
                          "-np", str(slots), "--api-key", API_KEY],
                         stdout=open(logpath, "w"), stderr=subprocess.STDOUT)
    for _ in range(360):
        if p.poll() is not None:
            tail = open(logpath).read().strip().splitlines()
            return None, "exited: " + (tail[-1][:200] if tail else "(no output)")
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=3).status_code == 200:
                return p, "healthy"
        except requests.RequestException:
            pass
        time.sleep(2)
    p.kill()
    return None, "timeout"

SERVER = SLOTS = FA = None
for fa in ("on", "auto"):
    for slots in range(MAX_SLOTS, 0, -1):
        p, why = try_serve(slots, fa)
        print(f"flash-attn={fa} slots={slots} total_ctx={slots * CTX_PER_SLOT} -> {why}")
        if p:
            SERVER, SLOTS, FA = p, slots, fa
            break
    if SERVER:
        break

if SERVER:
    print(f"\n>>> SERVING  slots={SLOTS}  flash-attn={FA}  port={PORT}")
    print(f">>> laptop flag:  --workers {SLOTS}")
else:
    print("\n>>> could not start; paste the last 'exited:' line above")

## 4. Public tunnel (`cloudflared`)

The server stays on `127.0.0.1`; `cloudflared` dials *out* to Cloudflare and routes a public URL
back down to this port. The URL is new every time this runs.

In [ ]:
import re, subprocess, time

if not os.path.exists("./cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

open("cloudflared.log", "w").close()
subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                 stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT)

BASE_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        BASE_URL = m.group(0)
        break
    time.sleep(2)

print("=" * 60)
print("BASE_URL =", BASE_URL)
print("API_KEY  =", API_KEY)
print("WORKERS  =", SLOTS)
print("=" * 60)

## 5. Verify the public endpoint (tunnel + auth + tool calls)

In [ ]:
import json, requests

hdr = {"Authorization": f"Bearer {API_KEY}"}
print("local  /health:", requests.get(f"http://127.0.0.1:{PORT}/health", timeout=10).json())
print("public /health:", requests.get(f"{BASE_URL}/health", timeout=30).json())
r = requests.get(f"{BASE_URL}/models", headers=hdr, timeout=30)
print("models (with key):", r.status_code, r.json()["data"][0]["id"])
print("models (no key)  :", requests.get(f"{BASE_URL}/models", timeout=30).status_code, "(401 expected)")

tools = [{"type": "function", "function": {
    "name": "add", "description": "Add two integers",
    "parameters": {"type": "object",
                   "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
                   "required": ["a", "b"]}}}]
body = {"model": MODEL, "messages": [{"role": "user", "content": "Use the add tool for 2 + 3."}],
        "tools": tools, "temperature": 0.0, "max_tokens": 256,
        "chat_template_kwargs": {"enable_thinking": True}}
r = requests.post(f"{BASE_URL}/chat/completions", json=body, headers=hdr, timeout=300).json()
print("tool_calls:", json.dumps(r["choices"][0]["message"].get("tool_calls"))[:250])

## 6. Run the grid on the laptop

Keep this tab open — the tunnel and server live only as long as this runtime. On the laptop,
**stop any running local batch first**, then (`WORKERS` and `MODEL` from the cells above):

```powershell
.\\.venv\\Scripts\\python.exe scripts\\run_batch.py --model gemma-4-E2B `
    --variants B0 M1 M2 M3 --thinking on --max-turns 15 --workers <WORKERS> `
    --base-url <BASE_URL> --api-key <API_KEY>
```

For E4B, set `MODEL = "E4B"` above and pass `--model gemma-4-E4B` (E4B is bigger: expect fewer
slots).

The grid is resumable: if this runtime dies, re-run the notebook and repeat the same command —
finished cells are skipped. Diagnostics: `server_<slots>_<fa>.log` and `cloudflared.log`.